# 08 · Main comparison on COCO (all methods, identical models)

**Purpose.** Every method works on the same detector candidates. Its operating point is selected on held-out (best-F1 and matched-recall), then evaluated unchanged on test, with image-level bootstrap 95 % CIs. Produces the main results table and the precision–recall frontier figure.

| | |
|---|---|
| **Proposal objective** | Objectives 2 and 3 |
| **Inputs** | all caches from notebooks 02, 03, 06, 07 |
| **Outputs** | `results/tables/main_comparison.{csv,md}`, `results/figures/*.png` |
| **Runtime (T4)** | ≈2–5 min (offline) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, load_splits
prepare_coco(cfg)          # after a runtime restart this re-extracts from the zip cached on Drive (~1-2 min)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.experiments import SplitData, available_methods, sweep, operating_point, evaluate_row, method_cost, DET_ONLY, PROPOSED
from dvd.metrics import decision_breakdown
from dvd.viz import plot_pr
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
MH, MT = available_methods(cfg, H), available_methods(cfg, T)
print("methods with complete caches:", list(MT))

In [ ]:
sweeps_h = {n: sweep(n, MH[n], H, cfg) for n in MH}
sweeps_t = {n: sweep(n, MT[n], T, cfg) for n in MT}
base_h = operating_point(sweeps_h[DET_ONLY]); ref_recall = base_h["recall"]
tau_ref = base_h["tau"]
scores_t = {i: T.ev.scores(i) for i in T.ids}
fp_base = evaluate_row(MT[DET_ONLY], T, base_h)["fp"]
rows = []
for crit in ["best_f1", "matched_recall"]:
    for n in MT:
        sel = operating_point(sweeps_h[n], crit, ref_recall)
        r = evaluate_row(MT[n], T, sel, n_boot=1000)
        b = decision_breakdown(T.labels, scores_t, r.pop("_masks"), tau_ref)
        rows.append({"criterion": crit, "method": n, **{k: r.get(k) for k in ["tau", "tau_low", "tau_high"]},
                     "recall_matched": sel.get("recall_matched", None),
                     "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "fp": r["fp"], "fp_per_image": r["fp_per_image"],
                     "fp_reduction": 1 - r["fp"] / fp_base, "f1_ci": r["ci"]["f1"], "precision_ci": r["ci"]["precision"],
                     "fp_removed_rate": b["fp_removed_rate"], "over_rejection_rate": b["over_rejection_rate"],
                     **method_cost(MT[n], T, r["n_verified"])})
main = pd.DataFrame(rows)
main[main.criterion == "best_f1"][["method", "precision", "recall", "f1", "fp_per_image", "fp_reduction", "over_rejection_rate", "total_latency_s"]]

In [ ]:
main[main.criterion == "matched_recall"][["method", "precision", "recall", "f1", "fp_per_image", "fp_reduction", "over_rejection_rate", "total_latency_s"]]

## Save the tables

In [ ]:
tdir = results_dir(cfg, "tables")
main.to_csv(tdir / "main_comparison.csv", index=False)
fmt = main.copy()
for c in ["precision", "recall", "f1", "fp_reduction", "fp_removed_rate", "over_rejection_rate"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:.3f}")
fmt["f1_ci"] = fmt["f1_ci"].map(lambda t: f"[{t[0]:.3f}, {t[1]:.3f}]")
cols = ["criterion", "method", "precision", "recall", "f1", "f1_ci", "fp_per_image", "fp_reduction", "over_rejection_rate", "total_latency_s", "total_invocations"]
(tdir / "main_comparison.md").write_text(fmt[cols].to_markdown(index=False, floatfmt=".3f"), encoding="utf-8")
print((tdir / "main_comparison.md").read_text())

## Precision–recall frontiers (test)

In [ ]:
sel = {n: main[(main.criterion == "best_f1") & (main.method == n)].iloc[0] for n in [DET_ONLY, PROPOSED] if n in MT}
plot_pr(sweeps_t, highlight={k: v for k, v in sel.items()}, title="All methods - test split")
plt.savefig(results_dir(cfg, "figures") / "pr_frontier_all_methods_test.png", dpi=150, bbox_inches="tight"); plt.show()

## Interpretation guide
- **Proposed vs Detector only** → does a verification stage reduce FPs at equal (matched) recall? (Objective 2)
- **Proposed vs Woodpecker-style** → same VLM, same candidates; difference = role separation + crop evidence. (Objective 3)
- **Proposed vs Global-view verifier + rule** → difference = crop-and-zoom evidence only.
- **Proposed vs Crop verifier decides alone** → difference = fusing detector confidence in a separate decision agent.
- **VCD / plain global yes/no** → a decoding-time remedy on global evidence.
- Overlapping bootstrap CIs mean the difference is not established on this test size; say so in the report.